# Face Matcher Unit Tests

These tests execute the actual `match_face_embedding` function from `face.py` without importing the Streamlit app, so they do not start the UI, access a webcam, or initialize the face detector. The test vectors are synthetic and do not require model downloads or image files.

In [1]:
import ast
import unittest
from pathlib import Path

import numpy as np

source_file = Path.cwd() / "face.py"
if not source_file.is_file():
    raise FileNotFoundError(f"Run this notebook from the project directory; not found: {source_file}")

syntax_tree = ast.parse(source_file.read_text(encoding="utf-8"))
matcher_node = next(
    node for node in syntax_tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "match_face_embedding"
)
matcher_module = ast.Module(body=[matcher_node], type_ignores=[])
matcher_namespace = {"np": np}
exec(compile(matcher_module, str(source_file), "exec"), matcher_namespace)
match_face_embedding = matcher_namespace["match_face_embedding"]

In [2]:
class MatchFaceEmbeddingTests(unittest.TestCase):
    def setUp(self):
        self.cache = {
            "Alice": {
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.6]], dtype=np.float32),
                "status": "Authorised",
            },
            "Bob": {
                "embeddings": np.array([[0.0, 1.0]], dtype=np.float32),
                "status": "Blacklisted",
            },
        }

    def test_returns_closest_identity_and_status(self):
        name, status, distance, similarity = match_face_embedding(
            np.array([0.99, 0.01], dtype=np.float32),
            self.cache,
            threshold=1.13,
        )

        self.assertEqual(name, "Alice")
        self.assertEqual(status, "Authorised")
        self.assertLess(distance, 0.03)
        self.assertGreater(similarity, 98.0)

    def test_rejects_match_beyond_threshold(self):
        far_cache = {
            "Alice": {
                "embeddings": np.array([[0.0, 1.0]], dtype=np.float32),
                "status": "Authorised",
            }
        }

        name, status, distance, similarity = match_face_embedding(
            np.array([1.0, 0.0], dtype=np.float32),
            far_cache,
            threshold=1.13,
        )

        self.assertIsNone(name)
        self.assertIsNone(status)
        self.assertAlmostEqual(distance, np.sqrt(2.0), places=5)
        self.assertEqual(similarity, 0.0)

    def test_empty_query_or_cache_returns_no_match(self):
        no_query = match_face_embedding(None, self.cache)
        no_cache = match_face_embedding(np.array([1.0, 0.0]), {})

        for result in (no_query, no_cache):
            self.assertIsNone(result[0])
            self.assertIsNone(result[1])
            self.assertTrue(np.isinf(result[2]))
            self.assertEqual(result[3], 0.0)

In [3]:
test_suite = unittest.defaultTestLoader.loadTestsFromTestCase(MatchFaceEmbeddingTests)
unittest.TextTestRunner(verbosity=2).run(test_suite)

test_empty_query_or_cache_returns_no_match (__main__.MatchFaceEmbeddingTests.test_empty_query_or_cache_returns_no_match) ... ok
test_rejects_match_beyond_threshold (__main__.MatchFaceEmbeddingTests.test_rejects_match_beyond_threshold) ... ok
test_returns_closest_identity_and_status (__main__.MatchFaceEmbeddingTests.test_returns_closest_identity_and_status) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.011s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

In [ ]:
import cv2

face_tree = ast.parse(source_file.read_text(encoding="utf-8"))
config_names = {"MODEL_NAME", "DETECTOR_BACKEND", "MATCH_THRESHOLD"}
config_nodes = [
    node for node in face_tree.body
    if isinstance(node, ast.Assign)
    and any(
        isinstance(target, ast.Name) and target.id in config_names
        for target in node.targets
    )
]
deepface_import = next(
    node for node in face_tree.body
    if isinstance(node, ast.ImportFrom)
    and node.module == "deepface"
    and any(alias.name == "DeepFace" for alias in node.names)
)
face_functions = [
    node for node in face_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name in {"extract_face_embedding", "match_face_embedding"}
]
face_runtime = {"np": np}
exec(
    compile(
        ast.Module(body=[*config_nodes, deepface_import, *face_functions], type_ignores=[]),
        str(source_file),
        "exec",
    ),
    face_runtime,
)

MODEL_NAME = face_runtime["MODEL_NAME"]
DETECTOR_BACKEND = face_runtime["DETECTOR_BACKEND"]
MATCH_THRESHOLD = face_runtime["MATCH_THRESHOLD"]
extract_face_embedding = face_runtime["extract_face_embedding"]
match_face_embedding = face_runtime["match_face_embedding"]

PHOTO_PATHS = {
    "Leo": Path("people/Leo.png"),
    "Brad": Path("people/Brad.jpg"),
    "me (Shang)": Path("people/Shang.jpg"),
}


class RegisteredPhotoPairTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.embeddings = {}
        for person, image_path in PHOTO_PATHS.items():
            if not image_path.is_file():
                raise FileNotFoundError(f"Photo not found: {image_path}")
            image = cv2.imread(str(image_path))
            embedding = extract_face_embedding(image)
            if embedding is None:
                raise ValueError(f"Could not extract a face embedding from {image_path}")
            cls.embeddings[person] = embedding

    def assert_pair_is_rejected(self, query_person, registered_person):
        registered_cache = {
            registered_person: {
                "embeddings": np.asarray(
                    [self.embeddings[registered_person]],
                    dtype=np.float32,
                ),
                "status": "Blacklisted",
            }
        }
        matched_name, _, distance, _ = match_face_embedding(
            self.embeddings[query_person],
            registered_cache,
            threshold=MATCH_THRESHOLD,
        )
        print(
            f"{query_person} vs {registered_person}: "
            f"{'MATCH' if matched_name else 'NO MATCH'}; "
            f"Euclidean L2={distance:.4f}; threshold={MATCH_THRESHOLD:.2f}"
        )
        self.assertIsNone(matched_name)
        self.assertGreater(distance, MATCH_THRESHOLD)

    def test_leo_does_not_match_brad(self):
        self.assert_pair_is_rejected("Leo", "Brad")

    def test_leo_does_not_match_me(self):
        self.assert_pair_is_rejected("Leo", "me (Shang)")

    def test_brad_does_not_match_me(self):
        self.assert_pair_is_rejected("Brad", "me (Shang)")


pair_test_suite = unittest.defaultTestLoader.loadTestsFromTestCase(RegisteredPhotoPairTests)
pair_test_result = unittest.TextTestRunner(verbosity=2).run(pair_test_suite)
if not pair_test_result.wasSuccessful():
    raise AssertionError("One or more registered-photo pair tests failed")